### Target Variable `scp_codes` has multiple levels of labeling

We want to limit our labels to diagnostic label superclasses (there are five): `NORM`, `MI`, `STTC`, `CD` and `HYP`.  
 * `NORM`: normal ECG
 * `MI`: myocardial infarction (heart-muscle damage from a lack of blood flow; aka "heart attack")
 * `STTC`: ST/T change; abnormalities in the ST segment or T waves (often linked to stress on the heart muscle like ischemia)
 * `CD`: conduction disturbance; problems with electical conduction through the heart
 * `HYP`: hypertrophy (thickening of heart wall or enlargement of a chamber)

This step will allow us to limit the scope of our analysis and prediction model to the five superclasses of cardiac diagnoses. 

#### Planned Pipeline: 
1. parse `scp_codes` into a map using file "scp_statements.csv"
2. only keep SCP-ECG codes that are present in file and have a DIAGNOSTIC class
3. map each SCP-ECG code to its superclass --> five binary targets (listed above)
* **Note**:records with no diagnostic code (only rhythm or waveform notations) will be dropped from this df

#### Still to decide?:
* **The likelihood threshold to use** (if at all).

Initially any presence of a dignostic label (regardless of the likelihood, even zero), will be maintained as "present" and count as a positive label; the documentation says that zero means "unknown" (not zero likelihood).  We may decide to set a minimum threshold at a later time to see how that affects the training and testing data balances.  For now, however, we will follow the benchmark example and use *any* presence of a diagnostic label , regardless of the recorded likelihood, as a positive label **Reference**: https://physionet.org/content/ptb-xl/1.0.3/example_physionet.py

In [1]:
# create Spark sesson and read in ptbxl_nw

from pyspark.sql import SparkSession
from pyspark.sql import functions as F  # keeps from overwriting Python functions like sum

spark = SparkSession.builder.getOrCreate()

# read in the saved parquet file
df_main = spark.read.parquet("/standard/ds7200-apt4c/team2/ptbxl_nw")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/21 14:24:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [2]:
df_main.select("ecg_id", "scp_codes").show(20, truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

+------+-----------------------------------------------------------------------------------+
|ecg_id|scp_codes                                                                          |
+------+-----------------------------------------------------------------------------------+
|2     |{'NORM': 80.0, 'SBRAD': 0.0}                                                       |
|4     |{'NORM': 100.0, 'SR': 0.0}                                                         |
|7     |{'NORM': 100.0, 'SR': 0.0}                                                         |
|8     |{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}                                             |
|10    |{'NORM': 100.0, 'SR': 0.0}                                                         |
|11    |{'NORM': 80.0, 'SARRH': 0.0}                                                       |
|14    |{'NORM': 100.0, 'SR': 0.0}                                                         |
|18    |{'AFLT': 100.0}                                               

### Parsing `scp_codes`

In [3]:
# preparing variable with substitution of double quote needed for JSON format of map

from pyspark.sql.types import MapType, StringType, DoubleType

# MapType() details the new schema 
# withColumns() adds output as new col
df_main = df_main.withColumn(
    "scp_map",
    F.from_json(F.regexp_replace("scp_codes", "'", '"'), MapType(StringType(), DoubleType()))
)
df_main.select("ecg_id", "scp_codes", "scp_map").show(5, truncate=False)

+------+--------------------------------------+--------------------------------------+
|ecg_id|scp_codes                             |scp_map                               |
+------+--------------------------------------+--------------------------------------+
|2     |{'NORM': 80.0, 'SBRAD': 0.0}          |{NORM -> 80.0, SBRAD -> 0.0}          |
|4     |{'NORM': 100.0, 'SR': 0.0}            |{NORM -> 100.0, SR -> 0.0}            |
|7     |{'NORM': 100.0, 'SR': 0.0}            |{NORM -> 100.0, SR -> 0.0}            |
|8     |{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}|{IMI -> 35.0, ABQRS -> 0.0, SR -> 0.0}|
|10    |{'NORM': 100.0, 'SR': 0.0}            |{NORM -> 100.0, SR -> 0.0}            |
+------+--------------------------------------+--------------------------------------+
only showing top 5 rows


**Notes**:scp_map is a structured type so we can ask for the keys (F.map_keys), do lookup or explode into new cols

---
#### Checking the Parsing and any Misalignment from CSV Read

In [4]:
# check that the rows parsed correctly (minimal mapped nulls)
df_main.filter(F.col("scp_codes").isNotNull() & F.col("scp_map").isNull()).count()

11

In [5]:
# checking null rows for misalignment during load or CSV read 
df_main.filter(F.col("scp_codes").isNotNull() & F.col("scp_map").isNull()) \
    .select("ecg_id", "scp_codes", "heart_axis", "infarction_stadium1", "strat_fold") \
    .show(11, truncate=60)

+------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+-------------+
|ecg_id|                                                   scp_codes|                                                  heart_axis|                                         infarction_stadium1|   strat_fold|
+------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+-------------+
|   640|                                                   Sokolow 2|                                                   4(NORM 50|                                                  SEHYP 15)"|         NULL|
|  1014| it may be due to sinus node dysfunction. holter monitor ...|{'AMI': 100.0, '1AVB': 100.0, 'NT_': 0.0, 'LPR': 0.0, 'SR...|                                              

In [6]:
base_xl = "/standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
ptbxl_fix = spark.read.csv(
    f"{base_xl}/ptbxl_database.csv",
    header=True, inferSchema=True,
    multiLine=True, quote='"', escape='"'
)
print(ptbxl_fix.count())
print(ptbxl_fix.filter(F.col("strat_fold").isNull()).count())

21799
0


In [7]:
ptbxl_fix.filter(F.col("ecg_id").isin(640, 1014, 10328)).select("ecg_id", "scp_codes", "heart_axis", "strat_fold").show(truncate=60)

+------+------------------------------------------------------------+----------+----------+
|ecg_id|                                                   scp_codes|heart_axis|strat_fold|
+------+------------------------------------------------------------+----------+----------+
|   640|      {'LVH': 80.0, 'SEHYP': 100.0, 'QWAVE': 0.0, 'SR': 0.0}|      NULL|         4|
|  1014|{'AMI': 100.0, '1AVB': 100.0, 'NT_': 0.0, 'LPR': 0.0, 'SR...|      NULL|        10|
| 10328|{'IMI': 15.0, 'LVH': 100.0, 'ISC_': 100.0, 'PVC': 100.0, ...|      NULL|         2|
+------+------------------------------------------------------------+----------+----------+



**A note about the success of the load and initial CSV read**: There are eleven rows that are incorrectly labeled due to anomalies in the `record` feature with its text.  We have made the decision not to patch this at this time (some analyses have already been built with this data).  11 rows out of 21795 is about 0.05% and we do not feel that this will impact our analysis.  We reserve the right to revisity this decision before submitting the project. 

In [8]:
# dropping the 11 problem rows (for now) 
df_main = df_main.filter(F.col("scp_map").isNotNull())
print(df_main.count())

21784


---

### Reading in Lookup Table --> Map Codes to Five Diagnostic Superclasses

In [9]:
ptbxl_baseaddress = "/standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

scp_lookup = spark.read.csv(f"{ptbxl_baseaddress}/scp_statements.csv", header=True, inferSchema=True)
scp_lookup.printSchema()
scp_lookup.show(10,truncate=False)

root
 |-- _c0: string (nullable = true)
 |-- description: string (nullable = true)
 |-- diagnostic: double (nullable = true)
 |-- form: double (nullable = true)
 |-- rhythm: double (nullable = true)
 |-- diagnostic_class: string (nullable = true)
 |-- diagnostic_subclass: string (nullable = true)
 |-- Statement Category: string (nullable = true)
 |-- SCP-ECG Statement Description: string (nullable = true)
 |-- AHA code: integer (nullable = true)
 |-- aECG REFID: string (nullable = true)
 |-- CDISC Code: string (nullable = true)
 |-- DICOM Code: string (nullable = true)

+-----+----------------------------------+----------+----+------+----------------+-------------------+---------------------------------------------------------+----------------------------------+--------+---------------------------+----------+----------+
|_c0  |description                       |diagnostic|form|rhythm|diagnostic_class|diagnostic_subclass|Statement Category                                       |SCP-ECG 

26/11/21 14:24:51 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , description, diagnostic, form, rhythm, diagnostic_class, diagnostic_subclass, Statement Category, SCP-ECG Statement Description, AHA code, aECG REFID, CDISC Code, DICOM Code
 Schema: _c0, description, diagnostic, form, rhythm, diagnostic_class, diagnostic_subclass, Statement Category, SCP-ECG Statement Description, AHA code, aECG REFID, CDISC Code, DICOM Code
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv


In [10]:
scp_lookup.select("diagnostic_class").distinct().show()

+----------------+
|diagnostic_class|
+----------------+
|              MI|
|             HYP|
|            NORM|
|            STTC|
|              CD|
|            NULL|
+----------------+



In [11]:
# looking at the number of null rows (rows wihtout a dignostic code)
scp_lookup.groupBy("diagnostic").agg(F.count("*").alias("num_rows"), F.sum(F.col("diagnostic_class").isNull().cast("int")).alias("nulls")).show()

# result: all 44 diagnostic codes map to a superclass

+----------+--------+-----+
|diagnostic|num_rows|nulls|
+----------+--------+-----+
|      NULL|      27|   27|
|       1.0|      44|    0|
+----------+--------+-----+



In [12]:
# building the lookup# looking at how many different diagnostic codes fall under each superclass label
lookup = (scp_lookup.filter(F.col("diagnostic") == 1.0).select(F.col("_c0").alias("diagnostic_code"),F.col("diagnostic_class").alias("superclass")))

lookup.groupBy("superclass").count().show()

+----------+-----+
|superclass|count|
+----------+-----+
|        MI|   14|
|       HYP|    5|
|      NORM|    1|
|      STTC|   13|
|        CD|   11|
+----------+-----+



**Notes**: Matching the lookup to the df needs tranformation of the df.  
* we do not preserve the likelihood value
* we explode, which makes an `ecg_id` have one row for each different diagnostic code it has (e.g., {IMI: 35, SR:0} will become two rows, one for each code

Then the INNER JOIN can attach the superclass code to each diagnostic_code it recognizes

In [13]:
codes = df_main.select("ecg_id", F.explode(F.map_keys("scp_map")).alias("diagnostic_code"))

labeled = codes.join(lookup, on="diagnostic_code", how="inner")
labeled.show(10)

26/11/21 14:24:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic, diagnostic_class
 Schema: _c0, diagnostic, diagnostic_class
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv


+---------------+------+----------+
|diagnostic_code|ecg_id|superclass|
+---------------+------+----------+
|           NORM|     2|      NORM|
|           NORM|     4|      NORM|
|           NORM|     7|      NORM|
|            IMI|     8|        MI|
|           NORM|    10|      NORM|
|           NORM|    11|      NORM|
|           NORM|    14|      NORM|
|           NORM|    21|      NORM|
|           NORM|    24|      NORM|
|           NORM|    25|      NORM|
+---------------+------+----------+
only showing top 10 rows


In [14]:
# how many ECGs have no diagnostic label (and therefore were dropped)? 
print(df_main.count(), labeled.select("ecg_id").distinct().count())

26/11/21 14:24:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic
 Schema: _c0, diagnostic
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv
[Stage 38:>                                                         (0 + 2) / 2]

21784 21378


In [15]:
# look at what dropped rows contain
unlabeled = df_main.join(labeled
        .select("ecg_id")
        .distinct(), on="ecg_id", how="left_anti")

unlabeled.select("scp_codes").groupBy("scp_codes").count().orderBy(F.desc("count")).show(10, truncate=False)

26/11/21 14:24:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic
 Schema: _c0, diagnostic
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv


+------------------------------------------+-----+
|scp_codes                                 |count|
+------------------------------------------+-----+
|{'PACE': 100.0}                           |267  |
|{'AFIB': 100.0}                           |40   |
|{'AFLT': 100.0}                           |17   |
|{'AFLT': 100.0, 'AFIB': 0.0}              |8    |
|{'PSVT': 100.0}                           |6    |
|{'AFLT': 100.0, 'ABQRS': 0.0, 'AFIB': 0.0}|5    |
|{'SR': 0.0}                               |5    |
|{'PVC': 100.0, 'BIGU': 100.0}             |5    |
|{'AFLT': 100.0, 'SVARR': 0.0}             |4    |
|{'AFLT': 100.0, 'ABQRS': 0.0}             |4    |
+------------------------------------------+-----+
only showing top 10 rows


**Decision** we will allow these values to drop and focus on the superclasses

In [16]:
# compact exploded rows into one single row per `ecg_id` with five binary columns
labels = (labeled.groupBy("ecg_id")
          .pivot("superclass", ["NORM", "MI", "STTC", "CD", "HYP"])
          .agg(F.max(F.lit(1))).fillna(0))

print(labels.count())
labels.show(5)

26/11/21 14:24:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic
 Schema: _c0, diagnostic
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv


21378


26/11/21 14:24:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic, diagnostic_class
 Schema: _c0, diagnostic, diagnostic_class
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv


+------+----+---+----+---+---+
|ecg_id|NORM| MI|STTC| CD|HYP|
+------+----+---+----+---+---+
|  5300|   1|  0|   0|  0|  0|
| 13285|   1|  0|   0|  1|  0|
| 18800|   0|  1|   1|  0|  1|
| 18024|   1|  0|   0|  0|  0|
|  1342|   0|  1|   0|  0|  1|
+------+----+---+----+---+---+
only showing top 5 rows


In [17]:
# attach labels to the full feature table df_main
ptb_superclass = df_main.join(labels, on="ecg_id", how="inner")

print(ptb_superclass.count(), len(ptb_superclass.columns))

26/11/21 14:25:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic
 Schema: _c0, diagnostic
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv


21378 2090


### Removing Unnecessary Columns

Including `height` which is ~68% nulls

In [18]:
ptb_superclass = ptb_superclass.drop("report", "scp_map", "height")
print(len(ptb_superclass.columns))

2087


### Rerunning Cecilia code for missing values: 

In [19]:
missing = ptb_superclass.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ptb_superclass.columns
])

missing_count = (missing.select(
    F.expr("stack({}, {})".format(
        len(ptb_superclass.columns),
        ", ".join([f"'{c}', `{c}`" for c in ptb_superclass.columns])))
    .alias("column", "missing_count"))
    .filter(F.col("missing_count") > 100)
    .orderBy("missing_count", ascending=False))

missing_count.show(100, truncate=False)

26/11/21 14:25:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/21 14:25:10 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic, diagnostic_class
 Schema: _c0, diagnostic, diagnostic_class
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv
26/11/21 14:25:21 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/11/21 14:25:47 WARN DAGScheduler: Broadcasting large task binary with size 5.0 MiB
[Stage 88:>                                                         (0 + 1) / 1]

+-----------------------+-------------+
|column                 |missing_count|
+-----------------------+-------------+
|pacemaker              |21365        |
|electrodes_problems    |21350        |
|unig_QRS_NotchAmp_V2   |21277        |
|unig_QRS_NotchOn_V2    |21277        |
|unig_QRS_NotchPeak_V2  |21277        |
|infarction_stadium2    |21275        |
|unig_QRS_NotchAmp_V1   |21274        |
|unig_QRS_NotchOn_V1    |21274        |
|unig_QRS_NotchPeak_V1  |21274        |
|unig_QRS_NotchAmp_V3   |21267        |
|unig_QRS_NotchOn_V3    |21267        |
|unig_QRS_NotchPeak_V3  |21267        |
|unig_QRS_NotchAmp_V4   |21109        |
|unig_QRS_NotchOn_V4    |21109        |
|unig_QRS_NotchPeak_V4  |21109        |
|burst_noise            |20797        |
|unig_QRS_NotchAmp_V5   |20431        |
|unig_QRS_NotchOn_V5    |20431        |
|unig_QRS_NotchPeak_V5  |20431        |
|baseline_drift         |19799        |
|extra_beats            |19498        |
|unig_QRS_NotchAmp_III  |19291        |


**Notes**: There are three different feature files combined into this 

### Writing a New Parquet File

This dataset now contains the superclasses we want to use for our targets. 

In [20]:
ptb_superclass.write.parquet("/standard/ds7200-apt4c/team2/ptb_superclass")

26/11/21 14:26:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , diagnostic, diagnostic_class
 Schema: _c0, diagnostic, diagnostic_class
Expected: _c0 but found: 
CSV file: file:///standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv
26/11/21 14:26:02 WARN DAGScheduler: Broadcasting large task binary with size 1086.7 KiB
                                                                                

### Looking at Nulls/Missing Values across Three Algorithms (sl12, ecgdeli, unig)

The features created by different algorithms are ***prefixed*** with: 
* `sl12_`
* `ecgdeli_`
* `unig_`